In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '16'
import sys
import json
import yaml
import argparse
import torch
import torch.nn as nn

from torchinfo import summary
import numpy as np
from datetime import datetime
from typing import Dict, List, Optional, Tuple
from tqdm import tqdm

# # 添加项目路径
# # sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

from Model import MultiModalTransformerV3
from Utils import (
    aggregate_metrics,
    MultiModalTrainingLoss,
    compute_metrics,
    get_confusion_matrix,
)
from Data_Pipeline.dataset import create_dataloaders
from Train.trainer import Trainer

from torch.amp import autocast

# 优化器
from Train.trainer import Trainer
from Train.train_utils import (
    setup_optimizer,
    setup_scheduler,
    setup_loss_functions,
    set_seed
)
import matplotlib.pyplot as plt
import torchinfo

In [18]:
# # 强制禁用 Flash Attention，使用传统实现
# os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
# # 关键：禁用 Flash Attention 内核
# torch.backends.cuda.enable_flash_sdp(False) 
# torch.backends.cuda.enable_mem_efficient_sdp(False)
# torch.backends.cuda.enable_math_sdp(True)

In [2]:
set_seed(42)

In [20]:
# model_v3 = MultiModalTransformerV3(**config['base_model'])

In [21]:
# import torch
# import torch.nn as nn
# from torchinfo import summary
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# # 3. 构造输入张量（维度需匹配配置）
# batch_size = 2
# time_steps = 3000  # LOB/Trade的时间步必须一致
# lob_input = torch.randn(batch_size, 4, time_steps, 10,device=device)  # (B,C,T,L)
# trade_input = torch.randn(batch_size, 12, time_steps,device=device)    # (B,F,T) 

# # 4. 封装模型：将字典输入转为位置参数（适配torchinfo）
# class WrappedMultiModalModel(nn.Module):
#     def __init__(self, original_model):
#         super().__init__()
#         self.original_model = original_model
    
#     def forward(self, lob, trade):
#         # 构造模型需要的字典输入
#         inputs = {"lob": lob,"trade":trade}
#         return self.original_model(inputs)

# # inputs = {'lob': lob_input}
# # with torch.no_grad():
# #     model(inputs)  # 这一步后，self.fusion 不再是 None
# wrapped_model = WrappedMultiModalModel(model_v3)
# trade_input = None
# # 5. 调用summary（核心：传入输入张量列表，顺序匹配封装模型的forward参数）
# summary(
#     wrapped_model,
#     input_data=[lob_input, trade_input],  # 先lob，后trade
#     col_names=["input_size", "output_size", "num_params", "trainable"],
#     col_width=20,
#     depth=5,  # 显示模型深度（层数）
#     device="cuda"  # 若用GPU，改为"cuda"（需确保张量在GPU上）
# )


In [22]:


# from Data_Pipeline.preprocessors.lob_data_process import process_lob_data
# from Data_Pipeline.preprocessors.trade_data_process import process_trade_data
# lob_data_dir = '/root/autodl-tmp/ETHUSDT/20levels_parquet'
# trade_data_dir = '/root/autodl-tmp/ETHUSDT/trade'


# date = ['2025-11-04','2025-11-05','2025-11-06','2025-11-07','2025-11-08','2025-11-09','2025-11-10',
#         '2025-11-11','2025-11-12','2025-11-13','2025-11-14','2025-11-15','2025-11-16','2025-11-17',
#         '2025-11-18','2025-11-19','2025-11-20','2025-11-21','2025-11-22','2025-11-23','2025-11-24',
#         '2025-11-25','2025-11-26','2025-11-27','2025-11-28','2025-11-29','2025-11-30','2025-12-01',
#         '2025-12-02','2025-12-03','2025-12-04','2025-12-05','2025-12-06','2025-12-07',
#         ]
# levels = 10


# lob_data = process_lob_data(data_dir=lob_data_dir,date = date,levels=levels)
# trade_data = process_trade_data(data_dir=trade_data_dir,date = date,window_ms=100)

# lob_data.write_parquet('/root/autodl-tmp/LOB_ETHUSDT_train.parquet')
# trade_data.write_parquet('/root/autodl-tmp/Trade_ETHUSDT_train_agg100ms.parquet')


In [3]:
# from Data_Pipeline.generators.label_gen import generate_data_dict,generate_ret_labels,align_trade_with_lob,generate_channel_data,generate_trade_labels
# lob_data_path = '/root/autodl-tmp/LOB_ETHUSDT_train.parquet'
# trade_data_path = '/root/autodl-tmp/Trade_ETHUSDT_train_agg100ms.parquet'

# label_window = 3000
# levels = 10
# data_dict,lob_labels, trade_labels= generate_data_dict(lob_data_path,trade_data_path,levels=levels,k=label_window)

# ## 存储为npy格式
# np.save('/root/autodl-tmp/train_data/lob_data.npy',data_dict['lob'])
# np.save('/root/autodl-tmp/train_data/trade_data.npy',data_dict['trade'])
# np.save('/root/autodl-tmp/train_data/lob_labels_ret.npy',lob_labels)
# np.save('/root/autodl-tmp/train_data/trade_labels_ret.npy',trade_labels)

In [3]:

class AblationExperimentRunner:
    """
    消融实验运行器
    
    负责：
    1. 加载配置
    2. 运行多个模型变体
    3. 收集和聚合结果
    4. 生成对比报告
    """
    
    def __init__(self, config_path: str, output_dir: Optional[str] = None):
        """
        初始化实验运行器
        
        Args:
            config_path: 实验配置文件路径
            output_dir: 输出目录 (可选，会覆盖配置中的路径)
        """
        # 加载配置
        with open(config_path, 'r') as f:
            self.config = yaml.safe_load(f)
        
        # 设置输出目录
        if output_dir:
            self.output_dir = output_dir
        else:
            self.output_dir = self.config.get('output', {}).get('results_dir', 'experiments/results')
        
        os.makedirs(self.output_dir, exist_ok=True)
        
        # # 初始化指标计算器
        # self.metrics_calculator = ExperimentMetrics()
        
        # 存储结果
        self.all_results = {}
        
        # 设备
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {self.device}")
    
    def _set_seed(self, seed: int):
        """设置随机种子以保证可复现性。"""
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
    
    def _create_model(self,variant_config: dict) -> nn.Module:
        """
        创建模型
        
        Args:
            variant_name: 变体名称
            variant_config: 变体配置
            
        Returns:
            model: 模型实例
        """
        base_model_config = self.config.get('base_model', {})
        
        # model = MultiModalTransformerV2(
        #     lob_config=base_model_config.get('lob_encoder', {}),
        #     trade_config=base_model_config.get('trade_encoder'),
        #     fusion_config=base_model_config.get('fusion', {}),
        #     backbone_config=base_model_config.get('transformer', {}),
        #     output_config=base_model_config.get('output_head', {}),
        #     use_cross_features=variant_config.get('use_cross_features', False),
        #     use_early_cross_attention=variant_config.get('use_early_cross_attention', False),
        #     use_hierarchical=variant_config.get('use_hierarchical', False),
        #     use_event_driven_trade=variant_config.get('use_event_driven_trade', False),
        #     use_mamba=variant_config.get('use_mamba', False),
        # )
        model = MultiModalTransformerV3(**base_model_config)
        # from Model import MultiModalTransformer
        # # 根据数据情况调整配置
        # lob_config = base_model_config.get('lob_encoder', {})
        # trade_config = base_model_config.get('trade_encoder') 
        # fusion_config = base_model_config.get('fusion', {})
        # transformer_config = base_model_config.get('transformer', {})
        # output_config = base_model_config.get('output_head', {})

        # model = MultiModalTransformer(
        #     lob_config=lob_config,
        #     trade_config=trade_config,
        #     fusion_config=fusion_config,
        #     transformer_config=transformer_config,
        #     output_config=output_config,
        #     use_revin=True
        # )
        
        return model.to(self.device)
    
    def _create_loss_fn(self, variant_config: dict) -> nn.Module:
        """创建损失函数"""
        loss_config = self.config.get('loss', {})
        
        use_contrastive = variant_config.get('use_contrastive_loss', False)
        
        if use_contrastive:
            return MultiModalTrainingLoss(
                num_classes=3,
                cls_weight=loss_config.get('cls_weight', 1.0),
                reg_weight=loss_config.get('reg_weight', 0.0),
                contrastive_weight=loss_config.get('contrastive_weight', 0.1),
                temporal_weight=loss_config.get('temporal_weight', 0.05),
                label_smoothing=loss_config.get('label_smoothing', 0.1),
                focal_gamma=loss_config.get('focal_gamma', 2.0),
                use_focal=loss_config.get('use_focal', True)
            )
        else:
            # variant_config['use_focal'] = False
            # org_loss_config  = self.config.get('org_loss', {})
            # device = 'cuda'
            # 设置损失函数
            loss_fn = setup_loss_functions(loss_config, device=self.device)
            # return FocalLoss(
            #     gamma=loss_config.get('focal_gamma', 2.0),
            #     alpha=loss_config.get('focal_alpha', 1.0),
            #     class_weights=loss_config.get('class_weights', None),
            #     # label_smoothing=loss_config.get('label_smoothing', 0.1)
            # )
            return loss_fn
    
    def _train_and_evaluate(
        self,
        model: nn.Module,
        train_loader,
        val_loader,
        loss_fn: nn.Module,
        optimizer,
        scheduler,
        variant_name = None,
        seed = None
    ) -> Dict:
        """
        训练并评估模型
        
        Args:
            model: 模型
            train_loader: 训练数据加载器
            val_loader: 验证数据加载器
            loss_fn: 损失函数
            variant_config: 变体配置
            seed: 随机种子
            
        Returns:
            result: 包含指标和训练历史的字典
        """

        
        # 创建训练器
        trainer = Trainer(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            loss_fn=loss_fn,
            optimizer=optimizer,
            scheduler=scheduler,
            config=self.config,
            device=self.device,
            variant_name=variant_name,
            seed=seed
        )
        # 开始训练
        print("=" * 50)
        history = trainer.fit()

        print("=" * 50)
        print("训练完成!")
        # print(f"最佳验证 F1 (Up/Down): {max(history['val_f1_updown']):.4f}")
        return history
    
    def _evaluate(self, model: nn.Module, data_loader) -> Dict:
        """评估模型"""
        model.eval()
        running_loss = 0.0
        all_preds = []
        all_labels = []
        all_probs = []
        pbar = tqdm(data_loader, desc='Validation', leave=False)
        with torch.no_grad():
            for inputs, labels, returns in pbar:
                # inputs, labels, _ = batch
                # inputs = {k: v.to(self.device) for k, v in inputs.items()}
                            # 移动数据
                if isinstance(inputs, dict):
                    inputs = {k: v.to(self.device, non_blocking=True) for k, v in inputs.items()}
                else:
                    inputs = inputs.to(self.device, non_blocking=True)
                labels = labels.to(self.device, non_blocking=True)
                returns = returns.to(self.device, non_blocking=True)
                            # 前向传播
                with autocast(device_type='cuda', enabled=True):
                    logits, _ = model(inputs)
                    # losses = self.loss_fn(cls_pred, labels, reg_pred, returns)
                # logits, _ = model(inputs)
                # probs = torch.softmax(logits, dim=-1)
                preds = torch.argmax(logits, dim=-1)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                # all_probs.append(probs.cpu().numpy())
        
        # all_preds = np.concatenate(all_preds)
        # all_labels = np.concatenate(all_labels)
        # all_probs = np.concatenate(all_probs)
        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)
        # all_probs = np.array(all_probs)
        metrics = compute_metrics(all_labels, all_preds)
        # metrics['loss'] = avg_loss
        # metrics.update(updown_metrics)
        metrics['confusion_matrix'] = get_confusion_matrix(all_labels, all_preds,normalize=False)
        
        return metrics

        # return self.metrics_calculator.compute_all_metrics(all_labels, all_preds, all_probs)
    
    def run_single_experiment(
        self,
        variant_name: str,
        variant_config: dict,
        data_dict: Dict[str, np.ndarray],
        labels: np.ndarray,
        returns: np.ndarray,
        seed: int
    ) -> Dict:
        """
        运行单次实验
        
        Args:
            variant_name: 变体名称
            variant_config: 变体配置
            data_dict: 数据字典
            labels: 标签
            returns: 收益率
            seed: 随机种子
            
        Returns:
            result: 实验结果
        """
        print(f"\n  Running {variant_name} with seed {seed}...")
        
        # 设置随机种子
        set_seed(seed)
        
        # 创建数据加载器
        data_config = self.config.get('data', {})
        dataloader_config = self.config.get('dataloader', {})
        training_config = self.config.get('training', {})

        train_loader, val_loader = create_dataloaders(
            data_dict=data_dict,
            labels=labels,
            returns=returns,
            config={'data': data_config, 'dataloader': dataloader_config},
            device=dataloader_config['device'] # 数据在CPU，训练时搬运
        )
        
        # 创建模型
        model = self._create_model(variant_config)
        ## 保存模型参数
        self._save_model(model,variant_name)
        # 创建损失函数
        loss_fn = self._create_loss_fn(variant_config)
        # print(f"loss_fn: {loss_fn}")
        # 优化器
        optimizer = setup_optimizer(model,training_config.get('optimizer', {}))
        # 学习率调度器
        scheduler = setup_scheduler(optimizer,training_config.get('scheduler', {}))
        # 训练和评估
        result = self._train_and_evaluate(
            model, 
            train_loader, 
            val_loader, 
            loss_fn,
            optimizer,
            scheduler,
            variant_name = variant_name,
            seed = seed
        )
        
        # # 计算效率指标
        # sample_input = next(iter(val_loader))[0]
        # sample_input = {k: v.to(self.device) for k, v in sample_input.items()}
        # efficiency_metrics = self.metrics_calculator.compute_efficiency_metrics(
        #     model, sample_input, num_warmup=5, num_runs=50
        # )
        # result['efficiency_metrics'] = efficiency_metrics
        
        return result
    
    def run_ablation_study(
        self,
        data_dict: Dict[str, np.ndarray],
        labels: np.ndarray,
        returns: np.ndarray,
        variants: Optional[List[str]] = None
    ) -> Dict:
        """
        运行消融实验
        
        Args:
            data_dict: 数据字典
            labels: 标签
            returns: 收益率
            variants: 要测试的变体列表 (None表示全部)
            
        Returns:
            all_results: 所有实验结果
        """
        seeds = self.config.get('seeds', [42])
        model_variants = self.config.get('model_variants', {})
        
        if variants is None:
            variants = list(model_variants.keys())
        
        for variant_name in variants:
            if variant_name not in model_variants:
                print(f"Warning: {variant_name} not found in config, skipping...")
                continue
            
            print(f"\n{'='*60}")
            print(f"Running experiments for: {variant_name}")
            print(f"{'='*60}")
            
            variant_config = model_variants[variant_name]
            seed_results = []
            
            for seed in seeds:
                # try:
                result = self.run_single_experiment(
                    variant_name, variant_config, data_dict, labels, returns, seed
                )
                print(f"confusion_matrix: {result['best_val_metrics']['confusion_matrix']}")
                seed_results.append(result)
                # except Exception as e:
                #     print(f"Error running {variant_name} with seed {seed}: {e}")
                #     continue
            
            if seed_results:
                # 聚合结果
                aggregated = self._aggregate_results(seed_results)
                self.all_results[variant_name] = aggregated
                
                # 保存中间结果
                self._save_results(f'ablation_results_partial.json')
        
        # 保存最终结果
        self._save_results('ablation_results_final.json')
        
        # 生成报告
        self._generate_report(variants)
        
        # return self.all_results
    
    def _aggregate_results(self, seed_results: List[Dict]) -> Dict:
        """聚合多次运行结果"""
        if not seed_results:
            return {}
        
        # 收集指标
        val_metrics_list = [r['best_val_metrics'] for r in seed_results if 'best_val_metrics' in r]
        
        aggregated = {
            'val_metrics': aggregate_metrics(val_metrics_list),
            # 'efficiency_metrics': seed_results[0].get('efficiency_metrics', {}),
            'best_epochs': [r.get('best_epoch', 0) for r in seed_results],
        }
        
        return aggregated
    
    def _save_results(self, filename: str):
        """保存结果"""
        filepath = os.path.join(self.output_dir, filename)
        with open(filepath, 'w') as f:
            json.dump(self.all_results, f, indent=2, default=str)
        print(f"Results saved to: {filepath}")
    
    def _generate_report(self, variants: List[str]):
        """生成实验报告"""
        report_lines = []
        report_lines.append("# 多模态LOB-Trade模型消融实验报告")
        report_lines.append(f"\n生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        
        report_lines.append(f"\n## 实验配置")
        report_lines.append(f"- Seeds: {self.config.get('seeds', [42])}")
        report_lines.append(f"- Epochs: {self.config.get('training', {}).get('epochs', 100)}")
        report_lines.append(f"- Batch Size: {self.config.get('dataloader', {}).get('batch_size', 256)}")
        
        # 主要指标对比表
        report_lines.append("\n## 主要指标对比\n")
        report_lines.append("| Model | Accuracy | F1-Macro | **F1-UpDown** | F1-Up | F1-Down | Dir-Acc | Signal-Q | recall_updown | precision_updown |")
        report_lines.append("|-------|----------|----------|---------------|-------|---------|---------|----------|--------------|----------------|")
        
        for variant in variants:
            if variant not in self.all_results:
                continue
            m = self.all_results[variant].get('val_metrics', {})
            
            def get_val(key):
                v = m.get(key, {})
                if isinstance(v, dict):
                    return f"{v.get('mean', 0):.4f}±{v.get('std', 0):.3f}"
                return f"{v:.4f}"
            
            report_lines.append(
                f"| {variant} | "
                f"{get_val('accuracy')} | "
                f"{get_val('f1_macro')} | "
                f"**{get_val('f1_updown')}** | "
                f"{get_val('f1_up')} | "
                f"{get_val('f1_down')} | "
                f"{get_val('directional_accuracy')} | "
                f"{get_val('signal_quality')} |"
                f"{get_val('precision_updown')} |"
                f"{get_val('recall_updown')} |"
            )
        
        # # 效率指标
        # report_lines.append("\n## 效率指标\n")
        # report_lines.append("| Model | Params (M) | Inference (ms) | GPU Mem (MB) |")
        # report_lines.append("|-------|------------|----------------|--------------|")
        
        # for variant in variants:
        #     if variant not in self.all_results:
        #         continue
        #     e = self.all_results[variant].get('efficiency_metrics', {})
        #     report_lines.append(
        #         f"| {variant} | "
        #         f"{e.get('total_params_M', 0):.2f} | "
        #         f"{e.get('inference_time_ms', 0):.2f} | "
        #         f"{e.get('gpu_memory_MB', 0):.1f} |"
        #     )
        
        # # 统计显著性
        # if 'M0_baseline' in self.all_results:
        #     report_lines.append("\n## 统计显著性检验 (vs Baseline)\n")
        #     report_lines.append("| Model | F1-UpDown Δ | p-value | Significant? |")
        #     report_lines.append("|-------|-------------|---------|--------------|")
            
        #     baseline_values = self.all_results['M0_baseline'].get('val_metrics', {}).get('f1_updown', {}).get('values', [])
            
        #     for variant in variants[1:]:
        #         if variant not in self.all_results:
        #             continue
        #         model_values = self.all_results[variant].get('val_metrics', {}).get('f1_updown', {}).get('values', [])
                
        #         if baseline_values and model_values:
        #             delta, p_value, is_sig = compute_significance_test(baseline_values, model_values)
        #             sig_mark = "✓" if is_sig else "✗"
        #             report_lines.append(f"| {variant} | {delta:+.4f} | {p_value:.4f} | {sig_mark} |")
        
        # 保存报告
        report_path = os.path.join(self.output_dir, 'experiment_report.md')
        with open(report_path, 'w') as f:
            f.write('\n'.join(report_lines))
        
        print(f"\n报告已保存至: {report_path}")

    def _save_model(self, model: nn.Module,variant_name: str):
        """保存模型参数"""


        # 3. 构造输入张量（维度需匹配配置）
        batch_size = 2
        time_steps = self.config.get('data', {}).get('history_T', 3000)  # LOB/Trade的时间步必须一致
        lob_dim = self.config.get('base_model', {}).get('lob_encoder', {}).get('in_channels', 4)
        trade_dim = self.config.get('base_model', {}).get('trade_encoder', {}).get('in_features', 12)
        lob_input = torch.randn(batch_size, lob_dim, time_steps, 10,device=self.device)  # (B,C,T,L) = (2,4,10,20)
        trade_input = torch.randn(batch_size, trade_dim, time_steps,device=self.device)    # (B,F,T) = (2,26,10)

        # 4. 封装模型：将字典输入转为位置参数（适配torchinfo）
        class WrappedMultiModalModel(nn.Module):
            def __init__(self, original_model):
                super().__init__()
                self.original_model = original_model
            
            def forward(self, lob, trade):
                # 构造模型需要的字典输入
                inputs = {"lob": lob,"trade":trade}
                return self.original_model(inputs)

        # inputs = {'lob': lob_input}
        # with torch.no_grad():
        #     model(inputs)  # 这一步后，self.fusion 不再是 None
        wrapped_model = WrappedMultiModalModel(model)
        model_summary_str = str(torchinfo.summary(
            wrapped_model,
            input_data=[lob_input, trade_input],
            col_names=["input_size", "output_size", "num_params", "trainable"],
            col_width=20,
            depth=4,
            device="cuda"
        ))

        # 2. 用 Matplotlib 绘制文本图
        plt.figure(figsize=(20, 25))  # 根据模型长度调整
        plt.text(0.01, 0.99, model_summary_str, fontsize=10, verticalalignment='top', family='monospace')
        plt.axis('off')
        plt.tight_layout()

        # 3. 保存图片
        plt.savefig(os.path.join(self.output_dir, f'{variant_name}_model_summary.png'), 
                    dpi=150, bbox_inches='tight')
        plt.close()
        # 5. 调用summary（核心：传入输入张量列表，顺序匹配封装模型的forward参数）,保存为图片
        # summary_img = summary(
        #     wrapped_model,
        #     input_data=[lob_input, trade_input],  # 先lob，后trade
        #     col_names=["input_size", "output_size", "num_params", "trainable"],
        #     col_width=20,
        #     depth=5,  # 显示模型深度（层数）
        #     device="cuda"  # 若用GPU，改为"cuda"（需确保张量在GPU上）
        # )
        # summary_img.savefig(os.path.join(self.output_dir, f'{variant_name}_model_summary.png'))




In [4]:


## 读取
lob_data = np.load('/root/autodl-tmp/train_data/lob_data.npy')
trade_data = np.load('/root/autodl-tmp/train_data/trade_data.npy')
lob_labels_ret = np.load('/root/autodl-tmp/train_data/lob_labels_ret.npy')
trade_labels_ret  = np.load('/root/autodl-tmp/train_data/trade_labels_ret.npy')

print(f"trade_data_agg.shape: {trade_data.shape}")
print(f"lob_data.shape: {lob_data.shape}")
print(f"lob_labels_ret.shape: {lob_labels_ret.shape}")

alpha= 0.001
labels_class = np.ones_like(lob_labels_ret, dtype=np.int8)
# 3. 向量化赋值：涨→2，跌→0
# 涨：lob_labels_ret > alpha
labels_class[lob_labels_ret > alpha] = 2
# 跌：lob_labels_ret < -alpha
labels_class[lob_labels_ret < -alpha] = 0



trade_data_agg.shape: (29369997, 12)
lob_data.shape: (29369997, 4, 10)
lob_labels_ret.shape: (29369997,)


In [5]:
np.unique(labels_class,return_counts = True)

(array([0, 1, 2], dtype=int8), array([ 6979616, 15706736,  6683645]))

In [27]:
# import polars as pl
# pl.DataFrame([lob_labels_ret,trade_labels_ret]).corr()

In [6]:
data_dict = {
    'lob': lob_data,
    'trade': None
}

labels = labels_class
returns = lob_labels_ret




In [7]:
variants = ['M0_baseline',
            ]

In [9]:
# 运行实验
config = '/root/lio/Trade_LOB_MultiModal/Configs/v3_config.yaml'
output_dir = '/root/lio/Trade_LOB_MultiModal/experiments/v3_results'
runner = AblationExperimentRunner(config, output_dir)


Using device: cuda


In [10]:
runner.run_ablation_study(data_dict, labels, returns, variants)


Running experiments for: M0_baseline

  Running M0_baseline with seed 42...
开始训练，共 50 个 epoch
设备: cuda
AMP: True
梯度累积步数: 1
--------------------------------------------------


Epoch 1/50 (101.8s)
  Train Loss: 0.3418, Train Acc: 0.5714
  Val Loss: 0.2903, Val Acc: 0.6303
  Val F1 (Macro): 0.5662
  Val Precision (Up/Down): 0.4980, Recall (Up/Down): 0.4899, F1 (Up/Down): 0.4939
  [NEW BEST] loss: 0.2903



Epoch 2/50 (100.9s)
  Train Loss: 0.3301, Train Acc: 0.5935
  Val Loss: 0.2872, Val Acc: 0.6391
  Val F1 (Macro): 0.5712
  Val Precision (Up/Down): 0.5113, Recall (Up/Down): 0.4824, F1 (Up/Down): 0.4964
  [NEW BEST] loss: 0.2872



KeyboardInterrupt: 